In [1]:
import pandas as pd
from datetime import datetime as dt
import xarray as xr
import re
import numpy as np
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

import pytz

In [2]:
# Time Zones

utc = pytz.UTC
akdt = pytz.timezone('America/Anchorage')

In [3]:
### Parameter Update for Figure Cleanup ###
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'figure.edgecolor': 'white',
    'figure.facecolor': 'white'
})

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans', 'Helvetica']

In [4]:
## Read in Station Data
stations = pd.read_csv('../../data_directory/metadata/ctd_stations_SKQ202310S_ToTS.txt', sep='\t')
station_lap = pd.read_csv('../../data_directory/metadata/station_metadata_laps.csv').drop(columns=['section_name_full', 'section', 'sectionStation', 'dist_km'])

stations[['station', 'cast']] = stations['StationCast'].str.extract(r'S(\d+)C(\d+)').astype(int)
stations = pd.merge(stations, station_lap[['station', 'lap']], on='station', how='left')

In [5]:
ice_data = pd.read_csv('../../data_directory/metadata/observed_ice_edge_bridgecam_updated072425.csv').rename(columns={
    'ice_obs': 'ice_class',
    'station': 'deployStation'
})
ice_data

,deployStation,ice_class
0,1,ow
1,2,ow
2,3,miz
3,4,miz
4,5,miz
...,...,...
295,296,ow
296,297,ow
297,298,ow
298,299,ow


In [6]:
### Read in trap metadata
traps = pd.read_csv('../../data_directory/metadata/trap_metadata.csv')[['Deployment', 'Deploy_Station', 'Recover_Station']]
traps.rename(columns={'Deployment': 'deployment', 'Deploy_Station': 'deployStation', 'Recover_Station': 'recoverStation'}, inplace=True)

traps.loc[traps['deployment'] == 'T026', 'deployStation'] = 171

trap_deploy = pd.merge(traps, station_lap, left_on='deployStation', right_on='station', how='left'
                       ).rename(columns={'latitude':'deployLat',
                                         'longitude':'deployLon',
                                         'lap':'deployLap'}
                        ).drop(columns=['station', 'cast'])

trap_recover = pd.merge(traps, station_lap, left_on='recoverStation', right_on='station', how='left'
                        ).rename(columns={'latitude':'recoverLat',
                                          'longitude':'recoverLon',
                                          'lap':'recoverLap'}
                                          ).drop(columns=['station', 'cast'])

trap_metadata = pd.merge(trap_deploy, trap_recover).merge(ice_data, on='deployStation', how='left')
trap_metadata

,deployment,deployStation,recoverStation,deployLap,deployLat,deployLon,recoverLap,recoverLat,recoverLon,ice_class
0,T001,15,15,NaN,70.8893,-161.571,NaN,70.8893,-161.571,ice
1,T001,15,15,NaN,70.8893,-161.571,NaN,70.8830,-161.354,ice
2,T001,15,15,NaN,70.8893,-161.571,NaN,70.8845,-161.310,ice
3,T001,15,15,NaN,70.8893,-161.571,NaN,70.8893,-161.571,ice
4,T001,15,15,NaN,70.8893,-161.571,NaN,70.8830,-161.354,ice
...,...,...,...,...,...,...,...,...,...,...
151,T028,198,252,7.0,71.4202,-164.403,8.0,71.0282,-163.009,ice
152,T029,207,251,7.0,70.9427,-165.189,8.0,70.8588,-164.225,ow
153,T029,207,251,7.0,70.9427,-165.189,8.0,70.8588,-164.225,ow
154,T029,207,251,7.0,70.9427,-165.189,8.0,70.8588,-164.225,ow


In [7]:
## Extract velocity data
velocity_data = pd.read_csv('../../data_directory/ADCP_vector_data/interp_uv_0_100_av_jie_mass_balance.dat', delim_whitespace=True, header=None)
velocity_data.columns = ['lon', 'lat', 'u', 'v']
velocity_data['u_deg_sec'] = velocity_data['u'] * (1e-5) / (111 * np.cos(np.deg2rad(velocity_data['lat'])))
velocity_data['v_deg_sec'] = velocity_data['v'] * (1e-5) / 111

northernmost_point = velocity_data.loc[velocity_data['lat'].idxmax()]
westernmost_point = velocity_data.loc[velocity_data['lon'].idxmin()]
southernmost_point = velocity_data.loc[velocity_data['lat'].idxmin()]
easternmost_point = velocity_data.loc[velocity_data['lon'].idxmax()]

hr_ends = pd.concat([northernmost_point.to_frame().T, westernmost_point.to_frame().T], ignore_index=True)
hre_ends = pd.concat([southernmost_point.to_frame().T, easternmost_point.to_frame().T], ignore_index=True)

/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/902718522.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  velocity_data = pd.read_csv('../../data_directory/ADCP_vector_data/interp_uv_0_100_av_jie_mass_balance.dat', delim_whitespace=True, header=None)


In [8]:
### Read in all GPS track data ###
def append_vars_from_trap_metadata(list_of_varNames, trapInteger, outputDataframe):
    for varName in list_of_varNames:
        value = trap_metadata[trap_metadata['deployment']==str('T'+f'{trapInteger:03}')][varName].values
        if len(value) > 0:
            outputDataframe[varName] = value[0]

list_of_vars = ['deployLap', 'recoverLap', 'ice_class']

In [9]:
# Dictionary to store all trap GPS data
all_trap_data = {}

In [10]:


def convert_akdt_to_utc(akdt_time):
    """Convert AKDT datetime to UTC"""
    if akdt_time.tzinfo is None:
        akdt_time = akdt.localize(akdt_time)
    return akdt_time.astimezone(utc)

def epoch_to_utc(epoch_seconds):
    """Convert epoch time to UTC datetime"""
    return utc.localize(dt.fromtimestamp(epoch_seconds))

# Read T010 spotter data
try:
    t010_spotter = pd.read_csv('../../data_directory/array_deployment_gps/T010_full_deployment_spotter.csv')
    append_vars_from_trap_metadata(list_of_vars, 10, t010_spotter)
    
    # Convert Epoch Time to UTC
    t010_spotter['time'] = t010_spotter['Epoch Time'].apply(epoch_to_utc)
    
    # Standardize column names
    t010_spotter['latitude'] = t010_spotter['Latitude (deg)']
    t010_spotter['longitude'] = t010_spotter['Longitude (deg)']
    
    all_trap_data['T010'] = t010_spotter
    print('T010 spotter data loaded')
except Exception as e:
    print(f'Error loading T010: {e}')

# Read all other trap deployments
for i in range(1, 30):
    if i == 1 or i == 10 or i == 21:
        continue  # Skip special cases
    
    try:
        df = pd.read_csv('../../data_directory/array_deployment_gps/T'+ f'{i:03}' +'_full_deployment.csv')
        df['trapNum'] = str('T' + f'{i:03}')
        append_vars_from_trap_metadata(list_of_vars, i, df)
        df.rename(columns={'Asset Name':'assetName',
                            'Data Date (AKDT)':'dateTimeAKDT',
                            ' Latitude':'latitude',
                            ' Longitude':'longitude'}, inplace=True)
        
        # Convert time column from AKDT to UTC
        df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])
        df['time'] = df['time_akdt'].apply(convert_akdt_to_utc)
        
        all_trap_data[f'T{i:03}'] = df
        print(f'T{i:03} read in successfully')
        
    except FileNotFoundError:
        print(f'T{i:03}_full_deployment.csv not found')
    except Exception as e:
        print(f'Error reading T{i:03}: {e}')

/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/229040472.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])
/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/229040472.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])
/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/229040472.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])
/var/folders/8g/k7dpf2yx1v31sdbm639f5

T010 spotter data loaded
T002 read in successfully
T003 read in successfully
T004 read in successfully
T005 read in successfully


/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/229040472.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])
/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/229040472.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])


T006 read in successfully
T007 read in successfully
T008 read in successfully
T009 read in successfully
T011 read in successfully
T012 read in successfully
T013 read in successfully
T014 read in successfully
T015 read in successfully
T016 read in successfully
T017 read in successfully
T018 read in successfully
T019 read in successfully
T020 read in successfully
Error reading T022: Naive time - no tzinfo set
T023 read in successfully
T024 read in successfully
T025 read in successfully
T026 read in successfully
T027 read in successfully
T028 read in successfully
T029 read in successfully


/var/folders/8g/k7dpf2yx1v31sdbm639f5tc00000gn/T/ipykernel_12074/229040472.py:43: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_akdt'] = pd.to_datetime(df['dateTimeAKDT'])


In [11]:
all_trap_data['T005']

,assetName,Asset Id,dateTimeAKDT,latitude,longitude,Speed,Heading,Report Body,trapNum,deployLap,recoverLap,ice_class,time_akdt,time
0,iES 6210,3.000000e+14,6/30/23 05:44,70.779194,-160.462425,0,0,01-7B-89-E2-A1-D7-16-F7-B6-15,T005,1.0,2.0,miz,2023-06-30 05:44:00,2023-06-30 13:44:00+00:00
1,iES 6210,3.000000e+14,6/30/23 05:15,70.777177,-160.474141,0,0,01-7B-2F-37-45-B6-5E-7A-C4-3D,T005,1.0,2.0,miz,2023-06-30 05:15:00,2023-06-30 13:15:00+00:00
2,iES 6210,3.000000e+14,6/30/23 04:44,70.775911,-160.489397,0,0,01-7B-43-F5-88-0D-BA-F0-EE-8D,T005,1.0,2.0,miz,2023-06-30 04:44:00,2023-06-30 12:44:00+00:00
3,iES 6210,3.000000e+14,6/30/23 04:13,70.774634,-160.504546,0,0,01-7B-8A-6A-3F-50-AD-E7-4A-C5,T005,1.0,2.0,miz,2023-06-30 04:13:00,2023-06-30 12:13:00+00:00
4,iES 6210,3.000000e+14,6/30/23 03:44,70.773518,-160.519738,0,0,01-7B-89-A8-71-F0-6A-A3-32-26,T005,1.0,2.0,miz,2023-06-30 03:44:00,2023-06-30 11:44:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,iES 6210,3.000000e+14,6/24/23 23:14,70.813097,-163.233705,0,0,01-7B-FC-F5-6B-9F-E1-BD-9A-46,T005,1.0,2.0,miz,2023-06-24 23:14:00,2023-06-25 07:14:00+00:00
245,iES 6210,3.000000e+14,6/24/23 22:43,70.812002,-163.244433,0,0,01-7B-24-CE-5F-CB-D4-A4-DA-1E,T005,1.0,2.0,miz,2023-06-24 22:43:00,2023-06-25 06:43:00+00:00
246,iES 6210,3.000000e+14,6/24/23 22:14,70.810565,-163.253188,0,0,01-7B-3B-5D-BA-D3-FA-03-4A-4F,T005,1.0,2.0,miz,2023-06-24 22:14:00,2023-06-25 06:14:00+00:00
247,iES 6210,3.000000e+14,6/24/23 21:44,70.809288,-163.262436,0,0,01-7B-C4-D8-A9-33-4A-92-D6-3E,T005,1.0,2.0,miz,2023-06-24 21:44:00,2023-06-25 05:44:00+00:00


In [12]:
# # Read T010 spotter data
# t010_spotter = pd.read_csv('../../data_directory/array_deployment_gps/T010_full_deployment_spotter.csv')
# append_vars_from_trap_metadata(list_of_vars, 10, t010_spotter)
# # t010_spotter['time'] = pd.to_datetime(t010_spotter['time'])
# all_trap_data['T010'] = t010_spotter

In [13]:
t010_spotter

,Battery Voltage (V),Power (W),Humidity (%rel),Epoch Time,Significant Wave Height (m),Peak Period (s),Mean Period (s),Peak Direction (deg),Peak Directional Spread (deg),Mean Direction (deg),...,Partition1 Mean Direction (deg),Partition1 Mean Directional Spread (deg),Mean Barometric Pressure (hPa),Processing Source,deployLap,recoverLap,ice_class,time,latitude,longitude
0,4.13,-0.05,3.2,1688534149,0.16,2.38,2.38,222.843,33.743,231.267,...,-,-,-,embedded,2.0,3.0,ow,2023-07-04 22:15:49+00:00,70.57455,-161.58075
1,4.13,-0.05,3.2,1688532349,0.15,2.26,2.28,217.653,38.614,231.454,...,-,-,-,embedded,2.0,3.0,ow,2023-07-04 21:45:49+00:00,70.57433,-161.59387
2,4.13,-0.03,3.2,1688530549,0.15,2.12,2.32,222.902,40.082,234.229,...,-,-,-,embedded,2.0,3.0,ow,2023-07-04 21:15:49+00:00,70.57417,-161.60735
3,4.13,-0.03,3.2,1688528749,0.13,3.20,2.26,238.188,33.695,233.014,...,-,-,-,embedded,2.0,3.0,ow,2023-07-04 20:45:49+00:00,70.57405,-161.62055
4,4.13,-0.03,3.2,1688526949,0.13,3.40,2.20,236.197,34.158,231.938,...,-,-,-,embedded,2.0,3.0,ow,2023-07-04 20:15:49+00:00,70.57392,-161.63373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,4.13,-0.09,5.6,1688118349,0.06,2.62,2.34,90.710,58.814,89.207,...,-,-,-,embedded,2.0,3.0,ow,2023-06-30 02:45:49+00:00,70.42032,-163.57108
232,4.16,0.00,5.6,1688116549,0.06,2.68,2.30,86.196,63.645,87.432,...,-,-,-,embedded,2.0,3.0,ow,2023-06-30 02:15:49+00:00,70.41875,-163.56870
233,4.16,0.00,5.6,1688114749,0.09,3.40,2.54,115.702,71.002,100.513,...,-,-,-,embedded,2.0,3.0,ow,2023-06-30 01:45:49+00:00,70.41747,-163.56642
234,4.13,-0.06,4.8,1688112949,0.09,2.92,2.36,84.100,50.072,81.490,...,-,-,-,embedded,2.0,3.0,ow,2023-06-30 01:15:49+00:00,70.41623,-163.56403


In [14]:
# # Read all other trap deployments
# for i in range(1, 30):
#     if i == 1 or i == 10 or i == 21:
#         continue  # Skip special cases
    
#     try:
#         df = pd.read_csv('../../data_directory/array_deployment_gps/T'+ f'{i:03}' +'_full_deployment.csv')
#         df['trapNum'] = str('T' + f'{i:03}')
#         append_vars_from_trap_metadata(list_of_vars, i, df)
#         df.rename(columns={'Asset Name':'assetName',
#                             'Data Date (AKDT)':'dateTimeAKDT',
#                             ' Latitude':'latitude',
#                             ' Longitude':'longitude'}, inplace=True)
        
#         # Convert time column to datetime
#         df['time'] = pd.to_datetime(df['dateTimeAKDT'])
        
#         all_trap_data[f'T{i:03}'] = df
#         print(f'T{i:03} read in successfully')
        
#     except FileNotFoundError:
#         print(f'T{i:03}_full_deployment.csv not found')
#     except Exception as e:
#         print(f'Error reading T{i:03}: {e}')

In [15]:
# ### Function to extract timestamp from SAR filename ###
# def extract_sar_timestamp(filename):
#     """
#     Extract timestamp from SAR filename
#     Assumes format includes YYYYMMDD_HHMMSS
#     Adjust the regex pattern based on your actual filename format
#     """
#     # Example for format: 0625_FullTrackHighRes_RCM3_OK2530212_PK2622395_1_SCLNB_20230625_173856_HH.nc
#     match = re.search(r'(\d{8})_(\d{6})', filename)
#     if match:
#         date_str = match.group(1)  # YYYYMMDD
#         time_str = match.group(2)  # HHMMSS
#         timestamp = dt.strptime(date_str + time_str, '%Y%m%d%H%M%S')
#         return timestamp
#     return None

In [16]:
### Function to extract timestamp from SAR filename ###
def extract_sar_timestamp(filename):
    """
    Extract timestamp from SAR filename (UTC)
    Assumes format includes YYYYMMDD_HHMMSS
    """
    # Example: 0625_FullTrackHighRes_RCM3_OK2530212_PK2622395_1_SCLNB_20230625_173856_HH.nc
    match = re.search(r'(\d{8})_(\d{6})', filename)
    if match:
        date_str = match.group(1)  # YYYYMMDD
        time_str = match.group(2)  # HHMMSS
        timestamp = dt.strptime(date_str + time_str, '%Y%m%d%H%M%S')
        # Localize to UTC since SAR timestamps are in Z time
        return utc.localize(timestamp)
    return None

In [17]:
### Function to find trap position at specific time ###
def get_trap_position_at_time(trap_data, target_time, time_window_hours=1):
    """
    Find the trap position closest to target_time
    
    Parameters:
    - trap_data: DataFrame with trap GPS data
    - target_time: datetime object for SAR image timestamp
    - time_window_hours: maximum time difference to consider (hours)
    
    Returns:
    - Dictionary with lat, lon, time if found, None otherwise
    """
    if 'time' not in trap_data.columns:
        return None
    
    # Calculate time differences
    time_diffs = abs(trap_data['time'] - target_time)
    min_diff_idx = time_diffs.idxmin()
    min_diff = time_diffs.loc[min_diff_idx]
    
    # Check if within time window
    if min_diff <= pd.Timedelta(hours=time_window_hours):
        row = trap_data.loc[min_diff_idx]
        return {
            'latitude': row['latitude'] if 'latitude' in row else row.get('Latitude (deg)'),
            'longitude': row['longitude'] if 'longitude' in row else row.get('Longitude (deg)'),
            'time': row['time'],
            'ice_class': row.get('ice_class', 'unknown')
        }
    
    return None


In [18]:
### Read SAR images ###
# ice_directory = '../../data_directory/SAR_ice_images/'
ice_directory = '../../../depreciated/SARSIC/high_res_full_track/'
# ice_directory = '../../../depreciated/SARSIC/netcdf_sar_data/'

In [19]:
# Get all SAR files
ice_files = [f for f in os.listdir(ice_directory) if f.endswith('.nc')]
ice_files.sort()

In [20]:
ice_files

['0620_FullTrackHighRes_RCM2_OK2313340_PK2613656_1_SCLNB_20230620_042817_HH.nc',
 '0625_FullTrackHighRes_RCM3_OK2530212_PK2622395_1_SCLNB_20230625_173856_HH.nc',
 '0628_FullTrackHighRes_RCM1_OK2313348_PK2625423_1_SCLNB_20230628_042718_HH.nc',
 '0629_FullTrackHighRes_RCM2_OK2313348_PK2626904_1_SCLNB_20230629_040433_HH.nc',
 '0630_FullTrackHighRes_RCM2_OK2313348_PK2628116_1_SCLNA_20230630_041227_HH.nc',
 '0701_FullTrackHighRes_RCM2_OK2417746_PK2629174_1_SCLNB_20230701_042018_HH.nc',
 '0701_FullTrackHighRes_RCM2_OK2496633_PK2629185_1_SCLNB_20230701_042018_HH.nc',
 '0702_FullTrackHighRes_RCM2_OK2417746_PK2630803_1_SCLNB_20230702_042818_HH.nc',
 '0708_FullTrackHighRes_RCM1_OK2417746_PK2640008_1_SCLNB_20230708_041225_HH.nc',
 '0710_FullTrackHighRes_RCM1_OK2417746_PK2642647_1_SCLNB_20230710_042719_HH.nc',
 '0713_FullTrackHighRes_RCM2_OK2417746_PK2646814_1_SCLNB_20230713_041940_HH.nc',
 '0714_FullTrackHighRes_RCM2_OK2417746_PK2648326_1_SCLNB_20230714_042740_HH.nc',
 '0717_FullTrackHighRes_RCM3

In [21]:
def read_in_SAR(file):
    with xr.open_dataset(file) as ds:
        ds = ds.isel(y=slice(None, None, 5), x=slice(None, None, 5))
        dataset = ds['band_1']
    return dataset

In [22]:
# Color mapping
ice_class_colors = {'ow': "#44C7FF", 'miz': "#FFA500", 'ice': "#F21A00"}
t010_color = '#44C7FF'
hr_color = "#2d981d"
hre_color = "#00098C"

# Plot parameters
text_fontsize = 55
track_endpoint_marker_size = 200
track_linewidth = 7.5
transect_linewidth = 7.5
extent = [-167, -160, 70, 72]

In [ ]:
### Create map for each SAR image ###
for sar_file in ice_files:
    print(f'\nProcessing {sar_file}')
    
    # Extract timestamp from filename
    sar_timestamp = extract_sar_timestamp(sar_file)
    if sar_timestamp is None:
        print(f'Could not extract timestamp from {sar_file}, skipping...')
        continue
    
    print(f'SAR timestamp: {sar_timestamp}')
    
    # Read SAR data
    sar_data = read_in_SAR(os.path.join(ice_directory, sar_file))
    
    # Find which traps were deployed at this time
    active_traps = {}
    for trap_name, trap_df in all_trap_data.items():
        position = get_trap_position_at_time(trap_df, sar_timestamp, time_window_hours=8)
        if position is not None:
            active_traps[trap_name] = position
            print(f'  {trap_name}: Position found at {position["time"]}')
    
    if len(active_traps) == 0:
        print(f'No active traps found for {sar_file}, skipping...')
        continue
    
    # Create figure
    plt.figure(figsize=(15., 15.), facecolor='none')
    ax = plt.axes(projection=ccrs.Orthographic(central_latitude=71, central_longitude=-164))
    ax.add_feature(cfeature.LAND, zorder=100)
    ax.add_feature(cfeature.COASTLINE, zorder=100)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=2, color='black', alpha=0.25, linestyle='--', draw_labels=False)
    ax.set_extent(extent, ccrs.PlateCarree())
    
    # Plot HR and HRE lines
    ax.plot(hr_ends['lon'], hr_ends['lat'], color=hr_color, transform=ccrs.PlateCarree(), linewidth=transect_linewidth)
    ax.plot(hre_ends['lon'], hre_ends['lat'], color=hre_color, transform=ccrs.PlateCarree(), linewidth=transect_linewidth)
    
    # Plot SAR imagery
    ax.pcolormesh(sar_data.lon.values,
                  sar_data.lat.values,
                  sar_data.values,
                  cmap='gray',
                  vmax=sar_data.mean()*2,
                  transform=ccrs.PlateCarree())
    
    # Plot trap positions and tracks
    for trap_name, position in active_traps.items():
        ice_class = position['ice_class']
        color = ice_class_colors.get(ice_class, 'gray')
        
        # Get the full trap dataset
        trap_df = all_trap_data[trap_name]
        
        # Filter to only points before or at the SAR timestamp
        historical_track = trap_df[trap_df['time'] <= sar_timestamp].copy()
        
        # Plot the track (all points prior to SAR time)
        if len(historical_track) > 1:
            ax.plot(historical_track['longitude'],
                    historical_track['latitude'],
                    color=color,
                    alpha=0.7,
                    linewidth=track_linewidth,
                    zorder=150,
                    transform=ccrs.PlateCarree())
    
    # Plot current position as a scatter point
    ax.scatter(position['longitude'],
               position['latitude'],
               s=track_endpoint_marker_size,
               color=color,
               alpha=1,
               zorder=200,
               transform=ccrs.PlateCarree())
    
    # Add label
    ax.text(position['longitude'],
            position['latitude'],
            trap_name,
            fontsize=text_fontsize,
            color=color,
            transform=ccrs.PlateCarree())
    
    # Save figure
    timestamp_str = sar_timestamp.strftime('%Y%m%d_%H%M%S')
    output_filename = f'../plot/collocated_ice/SAR_{timestamp_str}.png'
    plt.tight_layout()
    plt.savefig(output_filename, bbox_inches='tight', dpi=300, transparent=True)
    print(f'Saved: {output_filename}')
    plt.close()

print('\nProcessing complete!')

In [23]:
### Create map for each SAR image ###
for sar_file in ice_files:
    print(f'\nProcessing {sar_file}')
    
    # Extract timestamp from filename
    sar_timestamp = extract_sar_timestamp(sar_file)
    if sar_timestamp is None:
        print(f'Could not extract timestamp from {sar_file}, skipping...')
        continue
    
    print(f'SAR timestamp: {sar_timestamp}')
    
    # Read SAR data
    sar_data = read_in_SAR(os.path.join(ice_directory, sar_file))
    
    # Find which traps were deployed at this time
    active_traps = {}
    for trap_name, trap_df in all_trap_data.items():
        position = get_trap_position_at_time(trap_df, sar_timestamp, time_window_hours=8)
        if position is not None:
            active_traps[trap_name] = position
            print(f'  {trap_name}: Position found at {position["time"]}')
    
    if len(active_traps) == 0:
        print(f'No active traps found for {sar_file}, skipping...')
        continue
    
    # Create figure
    plt.figure(figsize=(15., 15.), facecolor='none')
    ax = plt.axes(projection=ccrs.Orthographic(central_latitude=71, central_longitude=-164))
    ax.add_feature(cfeature.LAND, zorder=100)
    ax.add_feature(cfeature.COASTLINE, zorder=100)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=2, color='black', alpha=0.25, linestyle='--', draw_labels=False)
    ax.set_extent(extent, ccrs.PlateCarree())
    
    # Plot HR and HRE lines
    ax.plot(hr_ends['lon'], hr_ends['lat'], color=hr_color, transform=ccrs.PlateCarree(), linewidth=transect_linewidth)
    ax.plot(hre_ends['lon'], hre_ends['lat'], color=hre_color, transform=ccrs.PlateCarree(), linewidth=transect_linewidth)
    
    # Plot SAR imagery
    ax.pcolormesh(sar_data.lon.values,
                  sar_data.lat.values,
                  sar_data.values,
                  cmap='gray',
                  vmax=sar_data.mean()*2,
                  transform=ccrs.PlateCarree())
    
    # Plot trap positions and tracks
    for trap_name, position in active_traps.items():
        ice_class = position['ice_class']
        color = ice_class_colors.get(ice_class, 'gray')
        
        # Get the full trap dataset
        trap_df = all_trap_data[trap_name]
        
        # Filter to only points before or at the SAR timestamp
        historical_track = trap_df[trap_df['time'] <= sar_timestamp].copy()
        
        # Plot the track (all points prior to SAR time)
        if len(historical_track) > 1:
            ax.plot(historical_track['longitude'],
                    historical_track['latitude'],
                    color=color,
                    alpha=0.7,
                    linewidth=track_linewidth,
                    zorder=150,
                    transform=ccrs.PlateCarree())
    
    # Plot current position as a scatter point
    ax.scatter(position['longitude'],
               position['latitude'],
               s=track_endpoint_marker_size,
               color=color,
               alpha=1,
               zorder=200,
               transform=ccrs.PlateCarree())
    
    # Add label
    ax.text(position['longitude'],
            position['latitude'],
            trap_name,
            fontsize=text_fontsize,
            color=color,
            transform=ccrs.PlateCarree())
    
    # Save figure
    timestamp_str = sar_timestamp.strftime('%Y%m%d_%H%M%S')
    output_filename = f'../plot/collocated_ice/SAR_{timestamp_str}.png'
    plt.tight_layout()
    plt.savefig(output_filename, bbox_inches='tight', dpi=300, transparent=True)
    print(f'Saved: {output_filename}')
    plt.close()

print('\nProcessing complete!')


Processing 0620_FullTrackHighRes_RCM2_OK2313340_PK2613656_1_SCLNB_20230620_042817_HH.nc
SAR timestamp: 2023-06-20 04:28:17+00:00
No active traps found for 0620_FullTrackHighRes_RCM2_OK2313340_PK2613656_1_SCLNB_20230620_042817_HH.nc, skipping...

Processing 0625_FullTrackHighRes_RCM3_OK2530212_PK2622395_1_SCLNB_20230625_173856_HH.nc
SAR timestamp: 2023-06-25 17:38:56+00:00
  T003: Position found at 2023-06-25 17:45:00+00:00
  T004: Position found at 2023-06-25 17:52:00+00:00
  T005: Position found at 2023-06-25 17:45:00+00:00
  T006: Position found at 2023-06-25 19:45:00+00:00


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:806: RuntimeWarning: invalid value encountered in disjoint
  return lib.disjoint(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:762: RuntimeWarning: invalid value encountered in covers
  return lib.covers(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:806: RuntimeWarning: invalid value encountered in disjoint
  return lib.disjoint(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:762: RuntimeWarning: invalid value encountered in covers
  return lib.covers(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib

Saved: ../plot/collocated_ice/SAR_20230625_173856.png

Processing 0628_FullTrackHighRes_RCM1_OK2313348_PK2625423_1_SCLNB_20230628_042718_HH.nc
SAR timestamp: 2023-06-28 04:27:18+00:00
  T003: Position found at 2023-06-28 04:15:00+00:00
  T005: Position found at 2023-06-28 04:14:00+00:00
  T006: Position found at 2023-06-28 04:30:00+00:00
  T007: Position found at 2023-06-28 04:23:00+00:00


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-p

Saved: ../plot/collocated_ice/SAR_20230628_042718.png

Processing 0629_FullTrackHighRes_RCM2_OK2313348_PK2626904_1_SCLNB_20230629_040433_HH.nc
SAR timestamp: 2023-06-29 04:04:33+00:00
  T005: Position found at 2023-06-29 04:15:00+00:00
  T006: Position found at 2023-06-29 04:00:00+00:00
  T007: Position found at 2023-06-29 03:51:00+00:00
  T008: Position found at 2023-06-29 04:00:18+00:00
  T009: Position found at 2023-06-29 10:14:08+00:00


...

/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-p

Saved: ../plot/collocated_ice/SAR_20230725_042019.png

Processing 0725_FullTrackHighRes_RCM2_OK2496633_PK2663544_1_SCLNB_20230725_042019_HH.nc
SAR timestamp: 2023-07-25 04:20:19+00:00
  T028: Position found at 2023-07-25 01:14:49+00:00
  T029: Position found at 2023-07-24 22:25:48+00:00


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)


Saved: ../plot/collocated_ice/SAR_20230725_042019.png

Processing 0726_FullTrackHighRes_RCM2_OK2417755_PK2665148_1_SCLNB_20230726_042819_HH.nc
SAR timestamp: 2023-07-26 04:28:19+00:00
No active traps found for 0726_FullTrackHighRes_RCM2_OK2417755_PK2665148_1_SCLNB_20230726_042819_HH.nc, skipping...

Processing complete!


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)


In [25]:
### Quick plots for T028 deployment on two specific SAR images ###

# SAR files to plot
t028_sar_files = [
    '0719_FullTrackHighRes_RCM3_OK2496633_PK2656157_1_SCLNB_20230719_173855_HH.nc',
    '0722_FullTrackHighRes_RCM1_OK2417755_PK2659573_1_SCLNB_20230722_042720_HH.nc'
]

# Get T028 data
if 'T028' in all_trap_data:
    t028_df = all_trap_data['T028']
    t028_deploy_lat = t028_df['latitude'].iloc[-1]  # Last row is first chronologically
    t028_deploy_lon = t028_df['longitude'].iloc[-1]
    
    print(f"\nT028 deployment position: {t028_deploy_lat:.4f}°N, {t028_deploy_lon:.4f}°W")
    
    # Create plots for each SAR image
    for sar_file in t028_sar_files:
        print(f'\nProcessing {sar_file} for T028')
        
        filepath = os.path.join(ice_directory, sar_file)
        
        # Extract SAR timestamp
        sar_timestamp = extract_sar_timestamp(sar_file)
        if sar_timestamp is None:
            print(f'Could not extract timestamp, skipping...')
            continue
        
        # Read SAR data
        try:
            sar_data = read_in_SAR(filepath)
        except Exception as e:
            print(f'Error reading SAR file: {e}')
            continue
        
        # Create figure
        plt.figure(figsize=(15., 15.), facecolor='none')
        ax = plt.axes(projection=ccrs.Orthographic(central_latitude=71, central_longitude=-164))
        ax.add_feature(cfeature.LAND, zorder=100)
        ax.add_feature(cfeature.COASTLINE, zorder=100)
        
        gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=2, color='black', alpha=0.25, 
                          linestyle='--', draw_labels=False)
        ax.set_extent(extent, ccrs.PlateCarree())
        
        # Plot SAR imagery
        ax.pcolormesh(sar_data.lon.values,
                      sar_data.lat.values,
                      sar_data.values,
                      cmap='gray',
                      vmax=sar_data.mean()*2,
                      transform=ccrs.PlateCarree())
        
        # Plot HR and HRE lines
        ax.plot(hr_ends['lon'], hr_ends['lat'], 
                color=hr_color, 
                transform=ccrs.PlateCarree(), 
                linewidth=transect_linewidth)
        ax.plot(hre_ends['lon'], hre_ends['lat'], 
                color=hre_color, 
                transform=ccrs.PlateCarree(), 
                linewidth=transect_linewidth)
        
        # Check if this is the 0722 image (show drift track)
        if '0722' in sar_file:
            # Filter to only points before or at the SAR timestamp
            historical_track = t028_df[t028_df['time'] <= sar_timestamp].copy()
            
            # Plot the drift track
            if len(historical_track) > 1:
                ax.plot(historical_track['longitude'],
                        historical_track['latitude'],
                        color=ice_class_colors['ice'],
                        alpha=0.7,
                        linewidth=track_linewidth,
                        zorder=150,
                        transform=ccrs.PlateCarree())
            
            # Get position at SAR time
            position = get_trap_position_at_time(t028_df, sar_timestamp, time_window_hours=8)
            if position is not None:
                current_lat = position['latitude']
                current_lon = position['longitude']
            else:
                # Fallback to nearest point
                current_lat = historical_track['latitude'].iloc[0]
                current_lon = historical_track['longitude'].iloc[0]
            
            # Plot current position
            ax.scatter(current_lon,
                       current_lat,
                       s=track_endpoint_marker_size,
                       color=ice_class_colors['ice'],
                       alpha=1,
                       zorder=200,
                       transform=ccrs.PlateCarree())
            
            # Add label at current position
            ax.text(current_lon,
                    current_lat,
                    'T028',
                    fontsize=text_fontsize,
                    color=ice_class_colors['ice'],
                    transform=ccrs.PlateCarree())
        else:
            # For 0719 image, just show deployment position
            ax.scatter(t028_deploy_lon,
                       t028_deploy_lat,
                       s=track_endpoint_marker_size,
                       color=ice_class_colors['ice'],
                       alpha=1,
                       zorder=200,
                       transform=ccrs.PlateCarree())
            
            # Add T028 label
            ax.text(t028_deploy_lon,
                    t028_deploy_lat,
                    'T028',
                    fontsize=text_fontsize,
                    color=ice_class_colors['ice'],
                    transform=ccrs.PlateCarree())
        
        # Save figure
        output_filename = f'../plot/collocated_ice/T028_{sar_file.replace(".nc", ".png")}'
        plt.tight_layout()
        plt.savefig(output_filename, bbox_inches='tight', dpi=300, transparent=True)
        print(f'Saved: {output_filename}')
        plt.close()
    
    print('\nT028 plots complete!')
else:
    print("\nWarning: T028 not found in all_trap_data")


T028 deployment position: 71.1833°N, -164.8108°W

Processing 0719_FullTrackHighRes_RCM3_OK2496633_PK2656157_1_SCLNB_20230719_173855_HH.nc for T028


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-p

Saved: ../plot/collocated_ice/T028_0719_FullTrackHighRes_RCM3_OK2496633_PK2656157_1_SCLNB_20230719_173855_HH.png

Processing 0722_FullTrackHighRes_RCM1_OK2417755_PK2659573_1_SCLNB_20230722_042720_HH.nc for T028


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)


Saved: ../plot/collocated_ice/T028_0722_FullTrackHighRes_RCM1_OK2417755_PK2659573_1_SCLNB_20230722_042720_HH.png

T028 plots complete!


/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/Users/jameslauer/miniconda3/envs/genericOcean/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
